# Project 2 — Digital Advertising Budget Optimizer
## Part 2: Big Data Processing & Performance Benchmarking
### Comparing Pandas vs. MapReduce vs. Apache Spark across 10K, 100K, 1M, and 10M Records

### 1. Context & Objectives
In Part 1, we engineered the advertising dataset schema and evaluated baseline performance with 10,000 records.
However, modern enterprise advertising platforms process millions of impression events daily. In Part 2, we evaluate how three different data processing paradigms handle scaling:
1. **Pandas**: Single-threaded in-memory processing (traditional data science).
2. **MapReduce**: Hadoop-style distributed streaming architecture (`mapper.py` -> shuffle & sort -> `reducer.py`).
3. **Apache Spark**: Distributed in-memory partitioned RDD processing with parallel execution across CPU cores.

#### Evaluation Dimensions:
- **Execution Runtime (Seconds)**: Processing speed and wall-clock time.
- **Memory Consumption (Peak RAM in MB)**: Memory efficiency and out-of-memory risk.
- **Throughput (Records/Second)**: Scalability when dataset size increases by 10x and 100x.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('../src')
print('Environment initialized.')

### 2. Loading Benchmark Results

In [ ]:
results_path = '../reports/big_data_benchmark_results.csv'
df_bench = pd.read_csv(results_path)
df_bench

### 3. Execution Runtime Analysis across Scales

In [ ]:
plt.figure(figsize=(10, 6))
palette = {'Pandas': '#2b5c8f', 'MapReduce': '#e76f51', 'Apache Spark': '#4ca585'}
df_actual = df_bench[df_bench['Size_Label'].isin(['10K', '100K', '1M'])]
ax = sns.barplot(data=df_actual, x='Size_Label', y='Runtime_Sec', hue='Engine', palette=palette)
plt.title('Execution Runtime Comparison (Lower is Better)', fontsize=14, fontweight='bold')
plt.ylabel('Runtime (Seconds)')
plt.xlabel('Dataset Scale')
plt.show()

### 4. Peak Memory Consumption (RAM)

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df_actual, x='Size_Label', y='Peak_RAM_MB', hue='Engine', palette=palette)
plt.title('Peak Memory Usage Comparison (MB)', fontsize=14, fontweight='bold')
plt.ylabel('Peak RAM (MB)')
plt.xlabel('Dataset Scale')
plt.show()

### 5. Big Data Scalability Trajectory (10K to 10M Records)

In [ ]:
plt.figure(figsize=(11, 6))
markers = {'Pandas': 'o', 'MapReduce': 's', 'Apache Spark': '^'}
for eng in ['Pandas', 'MapReduce', 'Apache Spark']:
    sub = df_bench[df_bench['Engine'] == eng]
    plt.plot(sub['Size_Label'], sub['Runtime_Sec'], marker=markers[eng], 
             linewidth=2.5, markersize=8, label=eng, color=palette[eng])
plt.title('Scalability Curve from 10K to 10 Million Records', fontsize=14, fontweight='bold')
plt.ylabel('Execution Time (Seconds)')
plt.xlabel('Dataset Size')
plt.legend(fontsize=12)
plt.show()

### 6. Architectural Takeaways & Synthesis

| Paradigm | Best Used For | Architectural Bottleneck | Big Data Viability |
| :--- | :--- | :--- | :--- |
| **Pandas** | Datasets < 2 GB that fit entirely in RAM | Single-threaded; OOM crashes on large files | Poor for massive streaming streams |
| **MapReduce** | Massive batch pipelines where memory is strictly constrained | Disk-spilling overhead & shuffle sorting delay | Highly resilient, low memory footprint |
| **Apache Spark** | Large-scale production analytics & iterative machine learning | Memory-intensive caching | **Superior for high-throughput Big Data** |